In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct

spark = SparkSession.builder.appName("StateOfData-SilverToGold").getOrCreate()

BUCKET = "tech-challenge-state-of-data"
SILVER_PATH = f"s3://{BUCKET}/silver/state_of_data/dados_harmonizados"
GOLD_PATH = f"s3://{BUCKET}/gold/state_of_data"

silver = spark.read.parquet(SILVER_PATH)


In [ ]:
gold_resumo = (
    silver.groupBy("periodo_pesquisa")
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)

gold_perfil = (
    silver.groupBy(
        "periodo_pesquisa", "cargo_atual", "nivel_carreira",
        "regiao", "modelo_trabalho"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)

gold_remuneracao = (
    silver.groupBy(
        "periodo_pesquisa", "faixa_salarial", "nivel_carreira"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)


In [ ]:
gold_mercado = (
    silver.groupBy(
        "periodo_pesquisa", "oportunidade_buscada",
        "regiao", "nivel_carreira"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)

gold_tecnologias = (
    silver.groupBy(
        "periodo_pesquisa", "cloud_dia_a_dia",
        "cloud_preferida"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)

gold_ia = (
    silver.groupBy(
        "periodo_pesquisa", "uso_ia_trabalho",
        "nivel_carreira", "regiao"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)


In [ ]:
gold_diversidade = (
    silver.groupBy(
        "periodo_pesquisa", "genero",
        "regiao", "nivel_carreira"
    )
    .agg(countDistinct("id_resposta").alias("quantidade_profissionais"))
)


In [ ]:
gold_resumo.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/resumo_pesquisa")
gold_perfil.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/perfil_profissionais")
gold_remuneracao.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/remuneracao")
gold_mercado.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/mercado_trabalho")
gold_tecnologias.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/tecnologias")
gold_ia.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/inteligencia_artificial")
gold_diversidade.write.mode("overwrite").partitionBy("periodo_pesquisa").parquet(f"{GOLD_PATH}/diversidade")


In [ ]:
print("Resumo")
gold_resumo.orderBy("periodo_pesquisa").show()

print("Perfil")
gold_perfil.show(10, truncate=False)

print("Remuneração")
gold_remuneracao.show(10, truncate=False)
